<a href="https://colab.research.google.com/github/shadesvinay01/AI-sales_copilot/blob/main/AI_Sales_Copilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ------------------- STEP 1: INSTALL DEPENDENCIES -------------------
!pip install -q streamlit pandas numpy scikit-learn langchain openai pyngrok
!pip install -q colab-xterm
!pip install -q streamlit pandas numpy scikit-learn pyngrok

print("✅ Dependencies installed!") # For terminal in Colab


✅ Dependencies installed!


In [ ]:
# ------------------- STEP 2: CREATE THE APPLICATION FILES -------------------

# Create sales_copilot.py file
%%writefile sales_copilot.py

import os
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime
import random
from sklearn.ensemble import RandomForestClassifier
import streamlit as st

Overwriting sales_copilot.py


In [ ]:

# ------------------------------------------------------------
# 1. LEAD SCORING MODEL
# ------------------------------------------------------------
class LeadScorer:
    """Predict which leads will convert"""

    def __init__(self):
        self.model = RandomForestClassifier(n_estimators=100)

    def predict_score(self, lead_data):
        """0-100 score based on engagement"""
        score = 0

        # Email engagement
        if lead_data.get('email_opens', 0) > 3: score += 20
        if lead_data.get('email_clicks', 0) > 1: score += 15
        if lead_data.get('reply_rate', 0) > 0.3: score += 25

        # Activity
        if lead_data.get('meetings', 0) > 0: score += 30
        if lead_data.get('website_visits', 0) > 5: score += 10

        # Time factor
        if lead_data.get('days_since_contact', 999) < 7: score += 10
        if lead_data.get('days_since_contact', 999) > 30: score -= 20

        return max(0, min(100, score))

# ------------------------------------------------------------
# 2. OUTREACH GENERATOR (WITHOUT API KEY)
# ------------------------------------------------------------
class OutreachGenerator:
    """Generate personalized messages using templates"""

    def generate_email(self, lead_name, company, pain_points, product):
        """Template-based email generation"""

        templates = [
            f"""Subject: Helping {company} with {pain_points[0] if pain_points else 'growth'}

Hi {lead_name},

I've been following {company}'s journey and noticed you might be facing challenges with {pain_points[0] if pain_points else 'scaling sales'}.

Our product, {product}, specifically helps companies like yours by automating outreach and identifying hot leads. We've helped similar startups increase their conversion rates by 40%.

Would you be open to a quick 15-minute call next week to share some specific ideas for {company}?

Best regards,
[Your Name]""",

            f"""Subject: Quick idea for {company}

Hi {lead_name},

I came across {company} and was impressed by your work in the space.

Given your focus on [industry], I thought you might be interested in how {product} helps sales teams prioritize leads and personalize outreach at scale.

Happy to share a few case studies if you're interested!

Cheers,
[Your Name]"""
        ]

        return random.choice(templates)

    def generate_linkedin(self, lead_name, company):
        """LinkedIn message templates"""

        templates = [
            f"Hi {lead_name}, I've been following {company}'s growth - impressive work! I specialize in helping B2B startups streamline their sales process. Would love to connect and learn more about what you're building.",

            f"Hi {lead_name}, came across your profile and noticed we share an interest in sales tech. I'm currently working on solutions for early-stage startups. Would be great to connect!"
        ]

        return random.choice(templates)

# ------------------------------------------------------------
# 3. DATA ANALYZER
# ------------------------------------------------------------
class DataAnalyzer:
    """Analyze leads and interactions"""

    def __init__(self):
        # Create sample data
        self.leads = pd.DataFrame({
            'id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
            'name': ['John Smith', 'Sarah Chen', 'Mike Johnson', 'Priya Patel', 'David Kim',
                     'Lisa Wong', 'Alex Rodriguez', 'Fatima Ahmed', 'Tom Brown', 'Elena Garcia'],
            'company': ['TechCorp', 'GrowthLabs', 'StartupHub', 'AIDynamics', 'CloudScale',
                        'DataFlow', 'MobileFirst', 'EcomPlus', 'SaaSify', 'DevOpsPro'],
            'email': ['john@techcorp.com', 'sarah@growthlabs.com', 'mike@startuphub.com',
                      'priya@aidynamics.com', 'david@cloudscale.com', 'lisa@dataflow.com',
                      'alex@mobilefirst.com', 'fatima@ecomplus.com', 'tom@saasify.com', 'elena@devopspro.com'],
            'email_opens': [5, 2, 0, 8, 3, 1, 4, 0, 6, 2],
            'email_clicks': [3, 0, 0, 5, 1, 0, 2, 0, 4, 1],
            'meetings': [1, 0, 0, 2, 0, 0, 0, 0, 1, 0],
            'website_visits': [12, 3, 1, 25, 5, 2, 8, 0, 15, 4],
            'days_since_contact': [2, 15, 45, 1, 7, 10, 5, 60, 3, 20],
            'industry': ['SaaS', 'AI/ML', 'E-commerce', 'AI/ML', 'Cloud',
                        'Data', 'Mobile', 'E-commerce', 'SaaS', 'DevOps'],
            'status': ['meeting_scheduled', 'contacted', 'cold', 'negotiation', 'follow_up',
                      'contacted', 'interested', 'cold', 'proposal', 'new']
        })

        self.scorer = LeadScorer()

        # Calculate scores
        scores = []
        for _, row in self.leads.iterrows():
            scores.append(self.scorer.predict_score(row.to_dict()))
        self.leads['score'] = scores

        # Calculate conversion probability
        self.leads['probability'] = (self.leads['score'] / 100) * 100

    def get_hot_leads(self, threshold=70):
        """Get leads with score > threshold"""
        return self.leads[self.leads['score'] > threshold]

    def get_lead_by_id(self, lead_id):
        """Get specific lead"""
        return self.leads[self.leads['id'] == lead_id].iloc[0]

# ------------------------------------------------------------
# 4. STREAMLIT DASHBOARD
# ------------------------------------------------------------
def main():
    st.set_page_config(page_title="AI Sales Copilot", layout="wide")

    # Custom CSS
    st.markdown("""
        <style>
        .main-header {
            font-size: 3rem;
            color: #FF4B4B;
            text-align: center;
            margin-bottom: 2rem;
        }
        .hot-lead {
            background: linear-gradient(90deg, #ff6b6b, #ff8e8e);
            padding: 1rem;
            border-radius: 10px;
            color: white;
        }
        </style>
    """, unsafe_allow_html=True)

    st.markdown('<h1 class="main-header">🚀 AI Sales Copilot for Startups</h1>', unsafe_allow_html=True)

    # Initialize data
    analyzer = DataAnalyzer()
    generator = OutreachGenerator()

    # Sidebar
    st.sidebar.image("https://via.placeholder.com/150x150.png?text=AI+Sales", width=150)
    st.sidebar.title("Navigation")
    page = st.sidebar.radio("Go to", ["📊 Dashboard", "🎯 Lead Scoring", "✉️ Outreach", "📈 Analytics"])

    if page == "📊 Dashboard":
        # Metrics row
        col1, col2, col3, col4 = st.columns(4)

        with col1:
            st.metric("Total Leads", len(analyzer.leads))
        with col2:
            hot_leads = len(analyzer.get_hot_leads())
            st.metric("Hot Leads (Score >70)", hot_leads)
        with col3:
            avg_score = analyzer.leads['score'].mean()
            st.metric("Average Score", f"{avg_score:.1f}")
        with col4:
            meetings = analyzer.leads[analyzer.leads['meetings'] > 0].shape[0]
            st.metric("Meetings Booked", meetings)

        # Hot leads section
        st.markdown("---")
        st.header("🔥 Hot Leads (Priority Today)")

        hot_leads = analyzer.get_hot_leads()
        if not hot_leads.empty:
            for _, lead in hot_leads.head(3).iterrows():
                with st.container():
                    st.markdown(f"""
                    <div class="hot-lead">
                        <h3>{lead['name']} - {lead['company']}</h3>
                        <p>Score: {lead['score']:.0f}/100 | Probability: {lead['probability']:.1f}% | Industry: {lead['industry']}</p>
                        <p>Last contact: {lead['days_since_contact']} days ago | Visits: {lead['website_visits']}</p>
                    </div>
                    """, unsafe_allow_html=True)
        else:
            st.info("No hot leads at the moment")

        # All leads table
        st.markdown("---")
        st.header("📋 All Leads")
        st.dataframe(
            analyzer.leads[['name', 'company', 'industry', 'score', 'probability', 'status']]
            .sort_values('score', ascending=False),
            use_container_width=True
        )

    elif page == "🎯 Lead Scoring":
        st.header("🎯 Lead Scoring & Prediction")

        col1, col2 = st.columns([1, 1])

        with col1:
            st.subheader("Lead Score Distribution")
            score_data = pd.DataFrame({
                'Score Range': ['0-20', '21-40', '41-60', '61-80', '81-100'],
                'Count': [
                    len(analyzer.leads[analyzer.leads['score'] <= 20]),
                    len(analyzer.leads[(analyzer.leads['score'] > 20) & (analyzer.leads['score'] <= 40)]),
                    len(analyzer.leads[(analyzer.leads['score'] > 40) & (analyzer.leads['score'] <= 60)]),
                    len(analyzer.leads[(analyzer.leads['score'] > 60) & (analyzer.leads['score'] <= 80)]),
                    len(analyzer.leads[analyzer.leads['score'] > 80])
                ]
            })
            st.bar_chart(score_data.set_index('Score Range'))

        with col2:
            st.subheader("Industry-wise Average Score")
            industry_scores = analyzer.leads.groupby('industry')['score'].mean().round(1)
            st.dataframe(industry_scores, use_container_width=True)

        st.markdown("---")
        st.subheader("Predict Lead Score (Try Yourself)")

        col1, col2, col3 = st.columns(3)

        with col1:
            email_opens = st.slider("Email Opens", 0, 20, 5)
            email_clicks = st.slider("Email Clicks", 0, 10, 2)

        with col2:
            meetings = st.slider("Meetings Booked", 0, 5, 0)
            visits = st.slider("Website Visits", 0, 50, 10)

        with col3:
            days = st.slider("Days Since Last Contact", 0, 90, 7)

        if st.button("Calculate Score"):
            test_lead = {
                'email_opens': email_opens,
                'email_clicks': email_clicks,
                'meetings': meetings,
                'website_visits': visits,
                'days_since_contact': days
            }

            scorer = LeadScorer()
            score = scorer.predict_score(test_lead)

            if score >= 70:
                st.success(f"🔥 HOT LEAD! Score: {score}/100")
            elif score >= 40:
                st.info(f"👍 Warm Lead - Follow Up. Score: {score}/100")
            else:
                st.warning(f"🧊 Cold Lead - Nurture. Score: {score}/100")

    elif page == "✉️ Outreach":
        st.header("✉️ Personalized Outreach Generator")

        col1, col2 = st.columns(2)

        with col1:
            st.subheader("📧 Email Generator")

            # Select lead
            lead_options = [f"{row['name']} - {row['company']}" for _, row in analyzer.leads.iterrows()]
            selected = st.selectbox("Select Lead", lead_options)

            if selected:
                lead_name = selected.split(" - ")[0]
                company = selected.split(" - ")[1]
                lead_data = analyzer.leads[analyzer.leads['name'] == lead_name].iloc[0]

                pain_points = st.text_input("Pain Points (comma separated)",
                                           "scaling sales, lead qualification, outreach automation")
                product = st.text_input("Your Product", "AI Sales Assistant")

                if st.button("Generate Email"):
                    email = generator.generate_email(lead_name, company,
                                                    [p.strip() for p in pain_points.split(',')],
                                                    product)
                    st.text_area("Generated Email:", email, height=300)

        with col2:
            st.subheader("🔗 LinkedIn Message")

            li_name = st.text_input("Contact Name (for LinkedIn)")
            li_company = st.text_input("Company")

            if st.button("Generate LinkedIn Message"):
                msg = generator.generate_linkedin(li_name, li_company)
                st.text_area("LinkedIn Message:", msg, height=150)

            st.markdown("---")
            st.subheader("💡 Outreach Tips")
            st.info("""
            • Personalize with specific company details
            • Reference recent news or achievements
            • Keep it under 150 words
            • Clear call-to-action
            • Follow up within 3-5 days
            """)

    elif page == "📈 Analytics":
        st.header("📈 Sales Analytics Dashboard")

        # Conversion funnel
        st.subheader("Conversion Funnel")

        funnel_data = pd.DataFrame({
            'Stage': ['Leads', 'Contacted', 'Interested', 'Meeting', 'Negotiation', 'Closed'],
            'Count': [
                len(analyzer.leads),
                len(analyzer.leads[analyzer.leads['status'] != 'cold']),
                len(analyzer.leads[analyzer.leads['score'] > 50]),
                len(analyzer.leads[analyzer.leads['meetings'] > 0]),
                len(analyzer.leads[analyzer.leads['status'] == 'negotiation']),
                2  # Sample closed deals
            ]
        })

        st.bar_chart(funnel_data.set_index('Stage'))

        # Metrics
        col1, col2, col3 = st.columns(3)

        with col1:
            conversion_rate = (2 / len(analyzer.leads)) * 100
            st.metric("Conversion Rate", f"{conversion_rate:.1f}%")

        with col2:
            avg_score = analyzer.leads['score'].mean()
            st.metric("Avg Lead Quality", f"{avg_score:.1f}/100")

        with col3:
            active = len(analyzer.leads[analyzer.leads['status'].isin(['meeting_scheduled', 'negotiation', 'proposal'])])
            st.metric("Active Deals", active)

        # Lead source analysis
        st.subheader("Performance by Industry")
        industry_perf = analyzer.leads.groupby('industry').agg({
            'score': 'mean',
            'meetings': 'sum',
            'name': 'count'
        }).round(1)
        industry_perf.columns = ['Avg Score', 'Total Meetings', 'Lead Count']
        st.dataframe(industry_perf, use_container_width=True)

if __name__ == "__main__":
    main()

print("✅ sales_copilot.py created successfully!")

# ------------------- STEP 3: RUN STREAMLIT IN COLAB -------------------

# Install ngrok for tunneling
!wget -q -nc https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip
!unzip -qq -n ngrok-stable-linux-amd64.zip

# Install colab-xterm for better terminal
!pip install -q colab-xterm
from google.colab import output
output.serve_kernel_port_as_window(8501)

# Run Streamlit in background
import threading
import time
import subprocess
import os

def run_streamlit():
    os.system("streamlit run sales_copilot.py --server.port 8501 &")

thread = threading.Thread(target=run_streamlit)
thread.start()

print("⏳ Starting Streamlit server...")
time.sleep(5)

print("\n" + "="*50)
print("✅ SUCCESS! AI Sales Copilot is running!")
print("="*50)
print("\n🔗 Click this link to open the dashboard:")
print("https://8501-{}-{}.colab.googleusercontent.com/".format(
    os.environ.get('COLAB_REPLICA_ID', 'xxxx'),
    os.environ.get('COLAB_DRIVE_ID', 'xxxx')
))
print("\n📱 Or open this URL manually:")
print("http://localhost:8501")
print("\n" + "="*50)

# ------------------- STEP 4: SHOW SAMPLE OUTPUT -------------------

print("\n📊 Sample Data Loaded:")
analyzer = DataAnalyzer()
print(f"Total Leads: {len(analyzer.leads)}")
print(f"Hot Leads: {len(analyzer.get_hot_leads())}")
print("\nTop 3 Leads:")
print(analyzer.leads.nlargest(3, 'score')[['name', 'company', 'score', 'probability']].to_string(index=False))

NameError: name 'st' is not defined

In [ ]:
%%writefile app.py

import pandas as pd
import numpy as np
import random
from datetime import datetime
import streamlit as st

# ------------------------------------------------------------
# 1. LEAD SCORING MODEL
# ------------------------------------------------------------
class LeadScorer:
    """Predict which leads will convert"""

    def predict_score(self, lead_data):
        """0-100 score based on engagement"""
        score = 0

        # Email engagement
        if lead_data.get('email_opens', 0) > 3: score += 20
        if lead_data.get('email_clicks', 0) > 1: score += 15
        if lead_data.get('reply_rate', 0) > 0.3: score += 25

        # Activity
        if lead_data.get('meetings', 0) > 0: score += 30
        if lead_data.get('website_visits', 0) > 5: score += 10

        # Time factor
        if lead_data.get('days_since_contact', 999) < 7: score += 10
        if lead_data.get('days_since_contact', 999) > 30: score -= 20

        return max(0, min(100, score))

# ------------------------------------------------------------
# 2. OUTREACH GENERATOR
# ------------------------------------------------------------
class OutreachGenerator:
    """Generate personalized messages using templates"""

    def generate_email(self, lead_name, company, pain_points, product):
        """Template-based email generation"""

        templates = [
            f"""Subject: Helping {company} with {pain_points[0] if pain_points else 'growth'}

Hi {lead_name},

I've been following {company}'s journey and noticed you might be facing challenges with {pain_points[0] if pain_points else 'scaling sales'}.

Our product, {product}, specifically helps companies like yours by automating outreach and identifying hot leads. We've helped similar startups increase their conversion rates by 40%.

Would you be open to a quick 15-minute call next week to share some specific ideas for {company}?

Best regards,
[Your Name]""",

            f"""Subject: Quick idea for {company}

Hi {lead_name},

I came across {company} and was impressed by your work in the space.

Given your focus on growth, I thought you might be interested in how {product} helps sales teams prioritize leads and personalize outreach at scale.

Happy to share a few case studies if you're interested!

Cheers,
[Your Name]"""
        ]

        return random.choice(templates)

    def generate_linkedin(self, lead_name, company):
        """LinkedIn message templates"""

        templates = [
            f"Hi {lead_name}, I've been following {company}'s growth - impressive work! I specialize in helping B2B startups streamline their sales process. Would love to connect and learn more about what you're building.",

            f"Hi {lead_name}, came across your profile and noticed we share an interest in sales tech. I'm currently working on solutions for early-stage startups. Would be great to connect!"
        ]

        return random.choice(templates)

# ------------------------------------------------------------
# 3. DATA ANALYZER
# ------------------------------------------------------------
class DataAnalyzer:
    """Analyze leads and interactions"""

    def __init__(self):
        # Create sample data
        self.leads = pd.DataFrame({
            'id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
            'name': ['John Smith', 'Sarah Chen', 'Mike Johnson', 'Priya Patel', 'David Kim',
                     'Lisa Wong', 'Alex Rodriguez', 'Fatima Ahmed', 'Tom Brown', 'Elena Garcia'],
            'company': ['TechCorp', 'GrowthLabs', 'StartupHub', 'AIDynamics', 'CloudScale',
                        'DataFlow', 'MobileFirst', 'EcomPlus', 'SaaSify', 'DevOpsPro'],
            'email': ['john@techcorp.com', 'sarah@growthlabs.com', 'mike@startuphub.com',
                      'priya@aidynamics.com', 'david@cloudscale.com', 'lisa@dataflow.com',
                      'alex@mobilefirst.com', 'fatima@ecomplus.com', 'tom@saasify.com', 'elena@devopspro.com'],
            'email_opens': [5, 2, 0, 8, 3, 1, 4, 0, 6, 2],
            'email_clicks': [3, 0, 0, 5, 1, 0, 2, 0, 4, 1],
            'meetings': [1, 0, 0, 2, 0, 0, 0, 0, 1, 0],
            'website_visits': [12, 3, 1, 25, 5, 2, 8, 0, 15, 4],
            'days_since_contact': [2, 15, 45, 1, 7, 10, 5, 60, 3, 20],
            'industry': ['SaaS', 'AI/ML', 'E-commerce', 'AI/ML', 'Cloud',
                        'Data', 'Mobile', 'E-commerce', 'SaaS', 'DevOps'],
            'status': ['meeting_scheduled', 'contacted', 'cold', 'negotiation', 'follow_up',
                      'contacted', 'interested', 'cold', 'proposal', 'new']
        })

        self.scorer = LeadScorer()

        # Calculate scores
        scores = []
        for _, row in self.leads.iterrows():
            scores.append(self.scorer.predict_score(row.to_dict()))
        self.leads['score'] = scores

        # Calculate conversion probability
        self.leads['probability'] = (self.leads['score'] / 100) * 100

    def get_hot_leads(self, threshold=70):
        """Get leads with score > threshold"""
        return self.leads[self.leads['score'] > threshold]

    def get_lead_by_id(self, lead_id):
        """Get specific lead"""
        return self.leads[self.leads['id'] == lead_id].iloc[0]

# ------------------------------------------------------------
# 4. STREAMLIT DASHBOARD
# ------------------------------------------------------------
def main():
    st.set_page_config(page_title="AI Sales Copilot", layout="wide")

    # Custom CSS
    st.markdown("""
        <style>
        .main-header {
            font-size: 3rem;
            color: #FF4B4B;
            text-align: center;
            margin-bottom: 2rem;
        }
        .hot-lead {
            background: linear-gradient(90deg, #ff6b6b, #ff8e8e);
            padding: 1rem;
            border-radius: 10px;
            color: white;
            margin: 10px 0;
        }
        .stApp {
            background-color: #f5f5f5;
        }
        </style>
    """, unsafe_allow_html=True)

    st.markdown('<h1 class="main-header">🚀 AI Sales Copilot for Startups</h1>', unsafe_allow_html=True)

    # Initialize data
    analyzer = DataAnalyzer()
    generator = OutreachGenerator()

    # Sidebar
    st.sidebar.image("https://via.placeholder.com/150x150.png?text=AI+Sales", width=150)
    st.sidebar.title("Navigation")
    page = st.sidebar.radio("Go to", ["📊 Dashboard", "🎯 Lead Scoring", "✉️ Outreach", "📈 Analytics"])

    if page == "📊 Dashboard":
        # Metrics row
        col1, col2, col3, col4 = st.columns(4)

        with col1:
            st.metric("Total Leads", len(analyzer.leads))
        with col2:
            hot_leads = len(analyzer.get_hot_leads())
            st.metric("Hot Leads (Score >70)", hot_leads)
        with col3:
            avg_score = analyzer.leads['score'].mean()
            st.metric("Average Score", f"{avg_score:.1f}")
        with col4:
            meetings = analyzer.leads[analyzer.leads['meetings'] > 0].shape[0]
            st.metric("Meetings Booked", meetings)

        # Hot leads section
        st.markdown("---")
        st.header("🔥 Hot Leads (Priority Today)")

        hot_leads = analyzer.get_hot_leads()
        if not hot_leads.empty:
            for _, lead in hot_leads.head(3).iterrows():
                st.markdown(f"""
                <div class="hot-lead">
                    <h3>{lead['name']} - {lead['company']}</h3>
                    <p>Score: {lead['score']:.0f}/100 | Probability: {lead['probability']:.1f}% | Industry: {lead['industry']}</p>
                    <p>Last contact: {lead['days_since_contact']} days ago | Visits: {lead['website_visits']}</p>
                </div>
                """, unsafe_allow_html=True)
        else:
            st.info("No hot leads at the moment")

        # All leads table
        st.markdown("---")
        st.header("📋 All Leads")
        st.dataframe(
            analyzer.leads[['name', 'company', 'industry', 'score', 'probability', 'status']]
            .sort_values('score', ascending=False),
            use_container_width=True
        )

    elif page == "🎯 Lead Scoring":
        st.header("🎯 Lead Scoring & Prediction")

        col1, col2 = st.columns([1, 1])

        with col1:
            st.subheader("Lead Score Distribution")
            score_data = pd.DataFrame({
                'Score Range': ['0-20', '21-40', '41-60', '61-80', '81-100'],
                'Count': [
                    len(analyzer.leads[analyzer.leads['score'] <= 20]),
                    len(analyzer.leads[(analyzer.leads['score'] > 20) & (analyzer.leads['score'] <= 40)]),
                    len(analyzer.leads[(analyzer.leads['score'] > 40) & (analyzer.leads['score'] <= 60)]),
                    len(analyzer.leads[(analyzer.leads['score'] > 60) & (analyzer.leads['score'] <= 80)]),
                    len(analyzer.leads[analyzer.leads['score'] > 80])
                ]
            })
            st.bar_chart(score_data.set_index('Score Range'))

        with col2:
            st.subheader("Industry-wise Average Score")
            industry_scores = analyzer.leads.groupby('industry')['score'].mean().round(1)
            st.dataframe(industry_scores, use_container_width=True)

        st.markdown("---")
        st.subheader("Predict Lead Score (Try Yourself)")

        col1, col2, col3 = st.columns(3)

        with col1:
            email_opens = st.slider("Email Opens", 0, 20, 5)
            email_clicks = st.slider("Email Clicks", 0, 10, 2)

        with col2:
            meetings = st.slider("Meetings Booked", 0, 5, 0)
            visits = st.slider("Website Visits", 0, 50, 10)

        with col3:
            days = st.slider("Days Since Last Contact", 0, 90, 7)

        if st.button("Calculate Score"):
            test_lead = {
                'email_opens': email_opens,
                'email_clicks': email_clicks,
                'meetings': meetings,
                'website_visits': visits,
                'days_since_contact': days
            }

            scorer = LeadScorer()
            score = scorer.predict_score(test_lead)

            if score >= 70:
                st.success(f"🔥 HOT LEAD! Score: {score}/100")
            elif score >= 40:
                st.info(f"👍 Warm Lead - Follow Up. Score: {score}/100")
            else:
                st.warning(f"🧊 Cold Lead - Nurture. Score: {score}/100")

    elif page == "✉️ Outreach":
        st.header("✉️ Personalized Outreach Generator")

        col1, col2 = st.columns(2)

        with col1:
            st.subheader("📧 Email Generator")

            # Select lead
            lead_options = [f"{row['name']} - {row['company']}" for _, row in analyzer.leads.iterrows()]
            selected = st.selectbox("Select Lead", lead_options)

            if selected:
                lead_name = selected.split(" - ")[0]
                company = selected.split(" - ")[1]
                lead_data = analyzer.leads[analyzer.leads['name'] == lead_name].iloc[0]

                pain_points = st.text_input("Pain Points (comma separated)",
                                           "scaling sales, lead qualification, outreach automation")
                product = st.text_input("Your Product", "AI Sales Assistant")

                if st.button("Generate Email"):
                    email = generator.generate_email(lead_name, company,
                                                    [p.strip() for p in pain_points.split(',')],
                                                    product)
                    st.text_area("Generated Email:", email, height=300)

        with col2:
            st.subheader("🔗 LinkedIn Message")

            li_name = st.text_input("Contact Name (for LinkedIn)")
            li_company = st.text_input("Company")

            if st.button("Generate LinkedIn Message"):
                msg = generator.generate_linkedin(li_name, li_company)
                st.text_area("LinkedIn Message:", msg, height=150)

            st.markdown("---")
            st.subheader("💡 Outreach Tips")
            st.info("""
            • Personalize with specific company details
            • Reference recent news or achievements
            • Keep it under 150 words
            • Clear call-to-action
            • Follow up within 3-5 days
            """)

    elif page == "📈 Analytics":
        st.header("📈 Sales Analytics Dashboard")

        # Conversion funnel
        st.subheader("Conversion Funnel")

        funnel_data = pd.DataFrame({
            'Stage': ['Leads', 'Contacted', 'Interested', 'Meeting', 'Negotiation', 'Closed'],
            'Count': [
                len(analyzer.leads),
                len(analyzer.leads[analyzer.leads['status'] != 'cold']),
                len(analyzer.leads[analyzer.leads['score'] > 50]),
                len(analyzer.leads[analyzer.leads['meetings'] > 0]),
                len(analyzer.leads[analyzer.leads['status'] == 'negotiation']),
                2  # Sample closed deals
            ]
        })

        st.bar_chart(funnel_data.set_index('Stage'))

        # Metrics
        col1, col2, col3 = st.columns(3)

        with col1:
            conversion_rate = (2 / len(analyzer.leads)) * 100
            st.metric("Conversion Rate", f"{conversion_rate:.1f}%")

        with col2:
            avg_score = analyzer.leads['score'].mean()
            st.metric("Avg Lead Quality", f"{avg_score:.1f}/100")

        with col3:
            active = len(analyzer.leads[analyzer.leads['status'].isin(['meeting_scheduled', 'negotiation', 'proposal'])])
            st.metric("Active Deals", active)

        # Lead source analysis
        st.subheader("Performance by Industry")
        industry_perf = analyzer.leads.groupby('industry').agg({
            'score': 'mean',
            'meetings': 'sum',
            'name': 'count'
        }).round(1)
        industry_perf.columns = ['Avg Score', 'Total Meetings', 'Lead Count']
        st.dataframe(industry_perf, use_container_width=True)

if __name__ == "__main__":
    main()

print("✅ app.py created successfully!")

# ------------------- STEP 3: RUN STREAMLIT -------------------

# Install localtunnel for sharing
!npm install -g localtunnel

# Run streamlit in background
import threading
import time
import subprocess
import os
import requests

def run_streamlit():
    os.system("streamlit run app.py --server.port 8501 > streamlit.log 2>&1 &")

# Start streamlit
thread = threading.Thread(target=run_streamlit)
thread.start()

print("⏳ Starting Streamlit server...")
time.sleep(5)

# Get public URL using localtunnel
try:
    # Get the public URL
    !lt --port 8501 > url.txt 2>&1 &
    time.sleep(3)

    # Read and display the URL
    with open('url.txt', 'r') as f:
        url = f.read().strip()
        print("\n" + "="*60)
        print("✅ AI SALES COPILOT IS RUNNING!")
        print("="*60)
        print("\n📱 OPEN THIS URL IN YOUR BROWSER:")
        print("\n🔗 https://8501-{}.loca.lt".format(os.environ.get('COLAB_REPLICA_ID', 'xxxx')))
        print("\n⚠️  Note: If asked, click 'Click to Continue'")
        print("\n" + "="*60)

except:
    print("\n" + "="*60)
    print("✅ AI SALES COPILOT IS RUNNING!")
    print("="*60)
    print("\n📱 OPEN THIS URL IN YOUR BROWSER:")
    print("\n🔗 http://localhost:8501")
    print("\n⚠️  Colab mein direct open nahi hoga")
    print("Alternative link: https://8501-{}.loca.lt".format(os.environ.get('COLAB_REPLICA_ID', 'xxxx')))
    print("\n" + "="*60)

# ------------------- STEP 4: SHOW SAMPLE DATA -------------------

print("\n📊 Sample Data Loaded:")
print("-" * 40)

# Create analyzer instance to show data
import pandas as pd
analyzer = DataAnalyzer()

print(f"Total Leads: {len(analyzer.leads)}")
print(f"Hot Leads: {len(analyzer.get_hot_leads())}")
print(f"Average Score: {analyzer.leads['score'].mean():.1f}/100")
print(f"Meetings Booked: {analyzer.leads[analyzer.leads['meetings'] > 0].shape[0]}")

print("\n🏆 Top 3 Hot Leads:")
print("-" * 40)
top_leads = analyzer.leads.nlargest(3, 'score')[['name', 'company', 'score', 'probability']]
for idx, lead in top_leads.iterrows():
    print(f"• {lead['name']} at {lead['company']}: Score {lead['score']:.0f}/100 | {lead['probability']:.1f}% probability")

print("\n" + "="*60)
print("✅ Setup Complete! Dashboard ready to use!")
print("="*60)

Overwriting app.py


In [ ]:
# CELL 1: Install Dependencies
!pip install -q streamlit pandas numpy scikit-learn pyngrok
print("✅ Dependencies installed successfully!")

✅ Dependencies installed successfully!


In [ ]:
# CELL 2: Create the main application file
%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import random
from datetime import datetime

# Lead Scorer Class
class LeadScorer:
    def predict_score(self, lead_data):
        score = 0
        if lead_data.get('email_opens', 0) > 3: score += 20
        if lead_data.get('email_clicks', 0) > 1: score += 15
        if lead_data.get('meetings', 0) > 0: score += 30
        if lead_data.get('website_visits', 0) > 5: score += 10
        if lead_data.get('days_since_contact', 999) < 7: score += 10
        return max(0, min(100, score))

# Outreach Generator
class OutreachGenerator:
    def generate_email(self, name, company, pain_points, product):
        return f"""Subject: Helping {company} with {pain_points[0]}

Hi {name},

I noticed {company} might be facing challenges with {pain_points[0]}. Our product {product} helps companies like yours automate outreach and increase conversions.

Would you be open to a quick 15-minute chat next week?

Best regards,
[Your Name]"""

    def generate_linkedin(self, name, company):
        return f"Hi {name}, I've been following {company}'s growth. I specialize in helping B2B startups streamline their sales process. Would love to connect!"

# Data Analyzer with Sample Data
class DataAnalyzer:
    def __init__(self):
        self.leads = pd.DataFrame({
            'id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
            'name': ['John Smith', 'Sarah Chen', 'Mike Johnson', 'Priya Patel', 'David Kim',
                     'Lisa Wong', 'Alex Rodriguez', 'Fatima Ahmed', 'Tom Brown', 'Elena Garcia'],
            'company': ['TechCorp', 'GrowthLabs', 'StartupHub', 'AIDynamics', 'CloudScale',
                        'DataFlow', 'MobileFirst', 'EcomPlus', 'SaaSify', 'DevOpsPro'],
            'email_opens': [5, 2, 0, 8, 3, 1, 4, 0, 6, 2],
            'email_clicks': [3, 0, 0, 5, 1, 0, 2, 0, 4, 1],
            'meetings': [1, 0, 0, 2, 0, 0, 0, 0, 1, 0],
            'website_visits': [12, 3, 1, 25, 5, 2, 8, 0, 15, 4],
            'days_since_contact': [2, 15, 45, 1, 7, 10, 5, 60, 3, 20],
            'industry': ['SaaS', 'AI/ML', 'E-commerce', 'AI/ML', 'Cloud',
                        'Data', 'Mobile', 'E-commerce', 'SaaS', 'DevOps'],
        })

        # Calculate scores
        scorer = LeadScorer()
        scores = []
        for _, row in self.leads.iterrows():
            scores.append(scorer.predict_score(row.to_dict()))
        self.leads['score'] = scores
        self.leads['probability'] = (self.leads['score'] / 100) * 100

    def get_hot_leads(self, threshold=70):
        return self.leads[self.leads['score'] > threshold]

# Streamlit Dashboard
st.set_page_config(page_title="AI Sales Copilot", layout="wide")

st.markdown("""
    <style>
    .main-header { font-size: 3rem; color: #FF4B4B; text-align: center; }
    .hot-lead { background: linear-gradient(90deg, #ff6b6b, #ff8e8e);
                padding: 1rem; border-radius: 10px; color: white; margin: 10px 0; }
    </style>
""", unsafe_allow_html=True)

st.markdown('<h1 class="main-header">🚀 AI Sales Copilot for Startups</h1>', unsafe_allow_html=True)

# Initialize
analyzer = DataAnalyzer()
generator = OutreachGenerator()

# Sidebar
st.sidebar.title("Navigation")
page = st.sidebar.radio("Go to", ["📊 Dashboard", "🎯 Lead Scoring", "✉️ Outreach"])

if page == "📊 Dashboard":
    # Metrics
    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Total Leads", len(analyzer.leads))
    col2.metric("Hot Leads", len(analyzer.get_hot_leads()))
    col3.metric("Avg Score", f"{analyzer.leads['score'].mean():.1f}")
    col4.metric("Meetings", analyzer.leads[analyzer.leads['meetings'] > 0].shape[0])

    # Hot Leads
    st.header("🔥 Hot Leads")
    hot_leads = analyzer.get_hot_leads()
    if not hot_leads.empty:
        for _, lead in hot_leads.head(3).iterrows():
            st.markdown(f"""
            <div class="hot-lead">
                <h3>{lead['name']} - {lead['company']}</h3>
                <p>Score: {lead['score']:.0f}/100 | Probability: {lead['probability']:.1f}%</p>
            </div>
            """, unsafe_allow_html=True)

    # All Leads
    st.header("📋 All Leads")
    st.dataframe(analyzer.leads[['name', 'company', 'industry', 'score', 'probability']])

elif page == "🎯 Lead Scoring":
    st.header("🎯 Lead Score Calculator")

    col1, col2 = st.columns(2)
    with col1:
        email_opens = st.slider("Email Opens", 0, 20, 5)
        email_clicks = st.slider("Email Clicks", 0, 10, 2)
    with col2:
        meetings = st.slider("Meetings", 0, 5, 0)
        visits = st.slider("Website Visits", 0, 50, 10)

    days = st.slider("Days Since Contact", 0, 90, 7)

    if st.button("Calculate Score"):
        test_lead = {
            'email_opens': email_opens,
            'email_clicks': email_clicks,
            'meetings': meetings,
            'website_visits': visits,
            'days_since_contact': days
        }
        scorer = LeadScorer()
        score = scorer.predict_score(test_lead)

        if score >= 70:
            st.success(f"🔥 HOT LEAD! Score: {score}/100")
        elif score >= 40:
            st.info(f"👍 Warm Lead - Score: {score}/100")
        else:
            st.warning(f"🧊 Cold Lead - Score: {score}/100")

elif page == "✉️ Outreach":
    st.header("✉️ Outreach Generator")

    col1, col2 = st.columns(2)

    with col1:
        st.subheader("Email Generator")
        name = st.text_input("Name")
        company = st.text_input("Company")
        pain = st.text_input("Pain Points", "scaling sales")
        product = st.text_input("Product", "AI Sales Assistant")

        if st.button("Generate Email"):
            email = generator.generate_email(name, company, [pain], product)
            st.text_area("Email:", email, height=250)

    with col2:
        st.subheader("LinkedIn Message")
        li_name = st.text_input("LinkedIn Name")
        li_company = st.text_input("LinkedIn Company")

        if st.button("Generate LinkedIn Message"):
            msg = generator.generate_linkedin(li_name, li_company)
            st.text_area("Message:", msg, height=150)

print("✅ app.py created successfully!")

Overwriting app.py


In [ ]:
# CELL 3: Start Streamlit Server
import threading
import time
import os

def run_streamlit():
    os.system("streamlit run app.py --server.port 8501 > /dev/null 2>&1 &")

thread = threading.Thread(target=run_streamlit)
thread.start()
time.sleep(3)

print("✅ Server started on port 8501")

✅ Server started on port 8501


In [ ]:
# CELL 4: Create Public URL with LocalTunnel
!npm install -g localtunnel > /dev/null 2>&1
!lt --port 8501 &

print("\n" + "="*60)
print("✅ AI SALES COPILOT IS READY!")
print("="*60)
print("\n📱 TO ACCESS THE DASHBOARD:")
print("1. Open this link: https://loca.lt")
print("2. Find your tunnel URL")
print("3. Click on the URL to open the dashboard")
print("\n🌐 Alternative: Use Colab's port forwarding")
print("   - Click the 🔗 icon in the right sidebar")
print("   - Enter port: 8501")
print("   - Click 'View port'")
print("\n" + "="*60)

your url is: https://nice-shoes-flash.loca.lt

✅ AI SALES COPILOT IS READY!

📱 TO ACCESS THE DASHBOARD:
1. Open this link: https://loca.lt
2. Find your tunnel URL
3. Click on the URL to open the dashboard

🌐 Alternative: Use Colab's port forwarding
   - Click the 🔗 icon in the right sidebar
   - Enter port: 8501
   - Click 'View port'



In [ ]:
# Running tunnels check
!ps aux | grep lt

root          88  0.0  0.0 1229368 4604 ?        Sl   18:54   0:00 /usr/local/bin/dap_multiplexer --domain_socket_path=/tmp/debugger_qbzabtghq
root        5215  0.0  0.0   7372  3460 ?        S    19:13   0:00 /bin/bash -c ps aux | grep lt
root        5217  0.0  0.0   6480  2476 ?        S    19:13   0:00 grep lt


In [ ]:
# finding password
!curl https://loca.lt/mytunnelpassword

35.186.160.75

In [ ]:
# CELL 1: Install and Run Everything (Single Cell)
!pip install -q streamlit pandas numpy

print("📦 Installing packages...")

# Create app.py file
with open('app.py', 'w') as f:
    f.write('''
import streamlit as st
import pandas as pd

st.set_page_config(page_title="AI Sales Copilot", layout="wide")
st.title("🚀 AI Sales Copilot for Startups")

# Sample data
df = pd.DataFrame({
    'name': ['John Smith', 'Sarah Chen', 'Priya Patel', 'Mike Johnson', 'Lisa Wong'],
    'company': ['TechCorp', 'GrowthLabs', 'AIDynamics', 'StartupHub', 'DataFlow'],
    'email_opens': [5, 8, 12, 2, 4],
    'meetings': [1, 2, 3, 0, 1],
    'score': [75, 92, 88, 45, 62]
})

# Metrics
col1, col2, col3 = st.columns(3)
col1.metric("Total Leads", len(df))
col2.metric("Hot Leads", len(df[df['score'] > 70]))
col3.metric("Avg Score", f"{df['score'].mean():.1f}")

# Table
st.subheader("📊 Leads Overview")
st.dataframe(df)

# Hot leads
st.subheader("🔥 Hot Leads")
hot_leads = df[df['score'] > 70]
for _, row in hot_leads.iterrows():
    st.success(f"**{row['name']}** at {row['company']} - Score: {row['score']}")
''')

print("✅ app.py created")

# Start streamlit
import subprocess
import time
import threading

def run_streamlit():
    subprocess.Popen(["streamlit", "run", "app.py", "--server.port=8501"],
                    stdout=subprocess.DEVNULL,
                    stderr=subprocess.DEVNULL)

thread = threading.Thread(target=run_streamlit)
thread.start()
print("🚀 Streamlit starting...")
time.sleep(3)

# Install and run localtunnel
!npm install -g localtunnel > /dev/null 2>&1
print("🔗 Creating tunnel...")
!lt --port 8501 &

print("\n" + "="*60)
print("✅ AI SALES COPILOT READY!")
print("="*60)
print("\n📱 OPEN THIS LINK IN BROWSER:")
print("   https://loca.lt")
print("\n📝 Tunnel URL will appear there - click it!")
print("\n💡 Alternative: Check Colab's port 8501 in right sidebar 🔗")
print("="*60)


📦 Installing packages...
✅ app.py created
🚀 Streamlit starting...
🔗 Creating tunnel...
your url is: https://stupid-frogs-add.loca.lt


In [1]:

import pandas as pd
import json
import requests

print("🚀 AI Sales Copilot - Loading from GitHub...")
print("="*50)

# GitHub raw content URLs
GITHUB_USER = "shadesvinay01"
REPO_NAME = "AI-sales_copilot"
BRANCH = "main"

# Sample files URLs
urls = {
    "leads": f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/data/sample_leads.csv",
    "leads_detailed": f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/data/sample_leads_detailed.csv",
    "conversations": f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/data/sample_conversations.json"
}

# Function to load data
def load_from_github(url, file_type):
    try:
        response = requests.get(url)
        if response.status_code == 200:
            if file_type == 'csv':
                return pd.read_csv(pd.compat.StringIO(response.text))
            elif file_type == 'json':
                return response.json()
        else:
            print(f"❌ Failed to load: {url}")
            return None
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

# Load leads data
print("\n📊 Loading sample_leads.csv...")
leads_df = load_from_github(urls["leads"], 'csv')
if leads_df is not None:
    print(f"✅ Loaded {len(leads_df)} leads")
    print("\nFirst 5 rows:")
    print(leads_df.head())

# Load detailed leads
print("\n📊 Loading sample_leads_detailed.csv...")
leads_detailed = load_from_github(urls["leads_detailed"], 'csv')
if leads_detailed is not None:
    print(f"✅ Loaded {len(leads_detailed)} detailed leads")

# Load conversations
print("\n💬 Loading sample_conversations.json...")
conversations = load_from_github(urls["conversations"], 'json')
if conversations is not None:
    print(f"✅ Loaded conversations for {len(conversations.get('leads', []))} leads")

print("\n" + "="*50)
print("✅ All sample data loaded successfully!")
print("="*50)

# Cell 2: Data analysis demo
if leads_df is not None:
    print("\n📈 Quick Analysis:")
    print(f"Total Leads: {len(leads_df)}")
    print(f"Average Score: {leads_df['score'].mean():.1f}")
    print(f"Hot Leads (score > 70): {len(leads_df[leads_df['score'] > 70])}")
    print(f"Industries: {', '.join(leads_df['industry'].unique())}")

# Cell 3: Streamlit dashboard with GitHub data
!pip install -q streamlit

%%writefile github_app.py
import streamlit as st
import pandas as pd
import requests

st.set_page_config(page_title="AI Sales Copilot - GitHub", layout="wide")

st.title("🚀 AI Sales Copilot - GitHub Version")

# GitHub URLs
GITHUB_USER = "shadesvinay01"
REPO_NAME = "AI-sales_copilot"

@st.cache_data
def load_github_data():
    url = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/main/data/sample_leads.csv"
    return pd.read_csv(url)

df = load_github_data()

st.subheader("📊 Leads from GitHub")
st.dataframe(df)

st.metric("Total Leads Loaded", len(df))

# Cell 4: Run Streamlit in Colab
!streamlit run github_app.py &
!npm install -g localtunnel
!lt --port 8501

🚀 AI Sales Copilot - Loading from GitHub...

📊 Loading sample_leads.csv...
❌ Failed to load: https://raw.githubusercontent.com/shadesvinay01/AI-sales_copilot/main/data/sample_leads.csv

📊 Loading sample_leads_detailed.csv...
❌ Failed to load: https://raw.githubusercontent.com/shadesvinay01/AI-sales_copilot/main/data/sample_leads_detailed.csv

💬 Loading sample_conversations.json...
❌ Failed to load: https://raw.githubusercontent.com/shadesvinay01/AI-sales_copilot/main/data/sample_conversations.json

✅ All sample data loaded successfully!
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 75.8 MB/s eta 0:00:00


UsageError: Line magic function `%%writefile` not found.


In [2]:
import requests

GITHUB_USER = "shadesvinay01"
REPO_NAME = "AI-sales_copilot"

# Try both main and master branches
for branch in ["main", "master"]:
    print(f"\n🔍 Checking branch: {branch}")
    api_url = f"https://api.github.com/repos/{GITHUB_USER}/{REPO_NAME}/git/trees/{branch}?recursive=1"
    response = requests.get(api_url)

    if response.status_code == 200:
        files = response.json().get('tree', [])
        print(f"✅ Found {len(files)} files/folders:")
        for file in files:
            if file['type'] == 'blob':
                print(f"  📄 {file['path']}")
        break
    else:
        print(f"❌ Branch '{branch}' not found")


🔍 Checking branch: main
✅ Found 8 files/folders:
  📄 AI_Sales_Copilot.ipynb
  📄 Data/sample_conversations.json
  📄 Data/sample_leads.csv
  📄 Data/sample_leads_detailed.csv
  📄 README.md
  📄 app.py
  📄 requirements.txt


In [4]:
# Cell 1: Corrected code with proper StringIO import
import pandas as pd
import json
import requests
from io import StringIO  # <-- IMPORTANT: StringIO ab yahan se import karo

print("🚀 AI Sales Copilot - Loading from GitHub...")
print("="*50)

GITHUB_USER = "shadesvinay01"
REPO_NAME = "AI-sales_copilot"
BRANCH = "main"

# Correct URLs with capital 'D'
urls = {
    "leads": f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/Data/sample_leads.csv",
    "leads_detailed": f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/Data/sample_leads_detailed.csv",
    "conversations": f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/Data/sample_conversations.json"
}

# Function to load data
def load_from_github(url, name):
    try:
        response = requests.get(url)
        if response.status_code == 200:
            if url.endswith('.csv'):
                # FIX: Using StringIO from io module
                df = pd.read_csv(StringIO(response.text))
                print(f"✅ Loaded {name}: {len(df)} rows")
                return df
            elif url.endswith('.json'):
                data = response.json()
                print(f"✅ Loaded {name}: JSON data")
                return data
        else:
            print(f"❌ Failed to load {name}: Status {response.status_code}")
            return None
    except Exception as e:
        print(f"❌ Error loading {name}: {e}")
        return None

# Load all files
leads_df = load_from_github(urls["leads"], "sample_leads.csv")
leads_detailed = load_from_github(urls["leads_detailed"], "sample_leads_detailed.csv")
conversations = load_from_github(urls["conversations"], "sample_conversations.json")

print("\n" + "="*50)
print("✅ Loading complete!")
print("="*50)

# Display sample data if loaded
if leads_df is not None:
    print("\n📊 sample_leads.csv preview:")
    print(leads_df.head())
    print(f"\n📈 Stats: {len(leads_df)} leads, Avg score: {leads_df['score'].mean():.1f}")

🚀 AI Sales Copilot - Loading from GitHub...
✅ Loaded sample_leads.csv: 10 rows
✅ Loaded sample_leads_detailed.csv: 10 rows
✅ Loaded sample_conversations.json: JSON data

✅ Loading complete!

📊 sample_leads.csv preview:
           name     company  email_opens  meetings  website_visits  \
0    John Smith    TechCorp            5         1              12   
1    Sarah Chen  GrowthLabs            8         2              25   
2   Priya Patel  AIDynamics           12         3              30   
3  Mike Johnson  StartupHub            1         0               3   
4     Lisa Wong    DataFlow            4         1               8   

     industry  score  
0        SaaS     75  
1       AI/ML     92  
2       AI/ML     88  
3  E-commerce     45  
4        Data     62  

📈 Stats: 10 leads, Avg score: 65.3


In [6]:
# Cell 1: Create github_app.py
%%writefile github_app.py

import streamlit as st
import pandas as pd
import requests
from io import StringIO

st.set_page_config(page_title="AI Sales Copilot", layout="wide")

st.title("🚀 AI Sales Copilot for Startups")
st.markdown("**Data loaded from GitHub**")

# GitHub URLs
GITHUB_USER = "shadesvinay01"
REPO_NAME = "AI-sales_copilot"
BRANCH = "main"

@st.cache_data
def load_data():
    url = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/Data/sample_leads.csv"
    return pd.read_csv(url)

@st.cache_data
def load_detailed():
    url = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/Data/sample_leads_detailed.csv"
    return pd.read_csv(url)

# Load data
df = load_data()
df_detailed = load_detailed()

# Sidebar
st.sidebar.title("Navigation")
page = st.sidebar.radio("Go to", ["📊 Dashboard", "📋 Detailed View"])

if page == "📊 Dashboard":
    # Metrics
    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Total Leads", len(df))
    col2.metric("Hot Leads", len(df[df['score'] > 70]))
    col3.metric("Avg Score", f"{df['score'].mean():.1f}")
    col4.metric("Industries", df['industry'].nunique())

    # Leads table
    st.subheader("📋 Leads Overview")
    st.dataframe(df, use_container_width=True)

    # Hot leads
    st.subheader("🔥 Hot Leads")
    hot_leads = df[df['score'] > 70]
    for _, row in hot_leads.iterrows():
        st.success(f"**{row['name']}** at {row['company']} - Score: {row['score']} | Industry: {row['industry']}")

else:
    st.subheader("📋 Detailed Lead Data")
    st.dataframe(df_detailed, use_container_width=True)

    # Download button
    csv = df_detailed.to_csv(index=False)
    st.download_button(
        label="📥 Download CSV",
        data=csv,
        file_name="leads_detailed.csv",
        mime="text/csv"
    )

st.markdown("---")
st.caption(f"Source: GitHub - {GITHUB_USER}/{REPO_NAME}")

Writing github_app.py


In [7]:
# Cell 2: Verify file exists
!ls -la github_app.py

-rw-r--r-- 1 root root 1909 Mar 20 03:22 github_app.py
